In [1]:
import os
import sqlite3

import nltk
import pandas as pd
import textgrid

from pathlib import Path
from typing import Any, List, Tuple

from nltk import pos_tag, word_tokenize
from nltk.corpus import cmudict
from nltk.stem import WordNetLemmatizer
from pandas import Series

In [2]:
def get_path(prev_folders: int = 0) -> str:
    """
    Identifies a specific directory according to the absolute path of the file.
    Args:
        prev_folders (int): An integer indicating the number of previous folders to get. If not given, 0 (current folder) is used by default.

    Returns:
        str: A text string with the identified path.
    """
    # Error handling
    if not isinstance(prev_folders, int) or prev_folders < 0:
        raise ValueError("prev_folders must be a non-negative integer")

    # Use pathlib to handle paths and directories
    path = Path.cwd()
    
    # Go up the specified number of directories
    for _ in range(prev_folders):
        path = path.parent
    
    # Use pathlib's as_posix method to get the string representation of the path
    # This automatically ensures the path ends with a slash
    path_str = path.as_posix() + '/'
    
    return path_str

In [3]:
def lemmatize_and_pos_tag(word: str, pos: str='v') -> Series:
    """
    Perform POS tagging on a word after lemmatization.

    Parameters:
    word (str): The word to be lemmatized and POS tagged.
    pos (str, optional): The part of speech category for lemmatization. Defaults to 'v' (verb).

    Returns:
    pd.Series: A pandas Series containing the lemmatized word and its POS tag.
    """
    lemma = lemmatizer.lemmatize(word, pos)

    word_pos = word_tokenize(lemma)
    word_pos = pos_tag(word_pos)
    return Series([lemma, word_pos])

In [4]:
def check_textgrid_files(folder_path: str) -> tuple:
    """
    Check all TextGrid files in a folder.

    Parameters:
    folder_path (str): The path to the folder containing the files.

    Returns:
    tuple: Two lists, the first one contains the names of the valid TextGrid files,
           the second one contains the names of the invalid TextGrid files.
    """
    valid_files = []
    invalid_files = []
    for file in os.listdir(folder_path):
        if file.endswith('.TextGrid'):
            file_path = os.path.join(folder_path, file)
            if is_valid_textgrid(file_path):
                valid_files.append(file)
            else:
                invalid_files.append(file)
    return valid_files, invalid_files

def is_valid_textgrid(file_path: str) -> bool:
    """
    Check if a file is a valid TextGrid file.

    Parameters:
    file_path (str): The path to the file.

    Returns:
    bool: True if the file is a valid TextGrid file, False otherwise.
    """
    with open(file_path, 'r') as f:
        first_line = f.readline().strip()
        second_line = f.readline().strip()
        third_line = f.readline().strip()  # This is the newline
        fourth_line = f.readline().strip()
    return (first_line == 'File type = "ooTextFile"' and 
            second_line == 'Object class = "TextGrid"' and 
            third_line == '' and
            fourth_line.startswith('xmin = '))

def load_single_dataframe(file_path: str, file_name: str) -> pd.DataFrame:
    """
    Load a single TextGrid file into memory as a pandas DataFrame.

    Parameters:
    file_path (str): The path to the file.
    file_name (str): The name of the file.

    Returns:
    df (pd.DataFrame): The loaded DataFrame.
    """
    tgrid = textgrid.read_textgrid(file_path)
    df = pd.DataFrame(tgrid)
    df['source'] = file_name
    return df

def load_dataframes(folder_path: str) -> pd.DataFrame:
    """
    Load all valid TextGrid files in a folder into memory as a single pandas DataFrame.

    Parameters:
    folder_path (str): The path to the folder containing the files.

    Returns:
    df (pd.DataFrame): The concatenated DataFrame.
    """
    valid_files, _ = check_textgrid_files(folder_path)
    df = pd.DataFrame()
    for file in valid_files:
        file_path = os.path.join(folder_path, file)
        df_name = os.path.splitext(file)[0]
        df_temp = load_single_dataframe(file_path, df_name)
        df = pd.concat([df, df_temp], ignore_index=True)
        print(f'DataFrame from {df_name} loaded into memory.')
        del df_temp
    return df

In [5]:
# nltk.download('cmudict')  # CMU Pronunciation Dictionary Corpus
PATH = get_path(1)
TEXTGRIDS_FOLDER = PATH + 'data/ds003020/derivative/TextGrids/'

lemmatizer = WordNetLemmatizer()

# Read data

In [6]:
# Archivos erroneos
good_files, bad_files = check_textgrid_files(TEXTGRIDS_FOLDER)
bad_files

['exorcism.TextGrid', 'legacy.TextGrid', 'life.TextGrid']

In [7]:
df = load_dataframes(TEXTGRIDS_FOLDER)

DataFrame from adollshouse loaded into memory.
DataFrame from adventuresinsayingyes loaded into memory.
DataFrame from afatherscover loaded into memory.
DataFrame from againstthewind loaded into memory.
DataFrame from alternateithicatom loaded into memory.
DataFrame from avatar loaded into memory.
DataFrame from backsideofthestorm loaded into memory.
DataFrame from becomingindian loaded into memory.
DataFrame from beneaththemushroomcloud loaded into memory.
DataFrame from birthofanation loaded into memory.
DataFrame from bluehope loaded into memory.
DataFrame from breakingupintheageofgoogle loaded into memory.
DataFrame from buck loaded into memory.
DataFrame from catfishingstrangerstofindmyself loaded into memory.
DataFrame from cautioneating loaded into memory.
DataFrame from christmas1940 loaded into memory.
DataFrame from cocoonoflove loaded into memory.
DataFrame from comingofageondeathrow loaded into memory.
DataFrame from eyespy loaded into memory.
DataFrame from firetestforlove

In [8]:
df.sample(n=15).head(15)

,start,stop,name,tier,source
630379,511.240590,511.729478,OUTSIDE,word,thumbsup
15926,261.987075,262.276417,UP,word,adventuresinsayingyes
351832,632.634694,632.684580,AH0,phone,mayorofthefreaks
676673,90.556689,90.616553,V,phone,whenmothersbullyback
693266,132.271882,132.331746,T,phone,wildwomenanddancingqueens
493894,142.109524,142.229252,M,phone,stagefright
151289,627.685941,627.715873,V,phone,comingofageondeathrow
449516,74.323583,74.363492,N,phone,reachingoutbetweenthebars
179956,443.704082,443.793878,D,phone,food
82639,38.365306,38.415193,V,phone,bluehope


In [9]:
df.shape

(697337, 5)

In [10]:
df['source'].value_counts()

source
breakingupintheageofgoogle    15498
mayorofthefreaks              14504
naked                         14028
thumbsup                      13938
haveyoumethimyet              13614
                              ...  
thetiniestbouquet              4347
myfathershands                 4309
jugglingandjesus               4171
againstthewind                 3829
sweetaspie                     3251
Name: count, Length: 81, dtype: int64

# Preprocessing

## Get phonetic transcription

In [11]:
# ARPAbet to IPA
cmu_dict = cmudict.dict()  # Loading the CMU pronunciation dictionary

df_phonetic = df.query("tier == 'phone'").copy()
df_phonetic.shape

(518506, 5)

## Get orthographic transcription

In [12]:
df['name'] = df['name'].str.lower()
df['tier'].value_counts()

tier
phone    518506
word     178831
Name: count, dtype: int64

In [13]:
df_orthographic = df.query("tier == 'word'").copy()
df_orthographic.shape

(178831, 5)

In [14]:
df_orthographic.sample(n=15).head(15)

,start,stop,name,tier,source
36229,375.229705,375.349433,and,word,alternateithicatom
219001,77.306803,77.586168,back,word,goingthelibertyway
221030,644.168481,644.417914,left,word,goingthelibertyway
51331,30.373469,30.463265,it,word,backsideofthestorm
175533,654.934014,655.073696,and,word,firetestforlove
402862,302.534921,302.783954,{br},word,naked
469655,3.225170,3.275057,sp,word,singlewomanseekingmanwich
219559,241.633333,241.884966,not,word,goingthelibertyway
143949,656.879592,657.228798,came,word,cocoonoflove
105876,627.735828,628.035147,only,word,breakingupintheageofgoogle


# NLP

In [15]:
df_orthographic[['lemma', 'pos_tagging']] = df_orthographic['name'].apply(lemmatize_and_pos_tag)

In [20]:
df_orthographic.sample(15)

,start,stop,name,tier,source,lemma,pos_tagging,is_adverb
298125,114.801587,114.941270,when,word,jugglingandjesus,when,"[(when, WRB)]",True
690914,349.378458,349.568027,up,word,wheretheressmoke,up,"[(up, RB)]",True
580010,113.863719,114.442404,even,word,theinterview,even,"[(even, RB)]",True
251134,663.095465,663.265079,don't,word,hangtime,don't,"[(do, VBP), (n't, RB)]",True
90021,171.672336,171.921769,even,word,bluehope,even,"[(even, RB)]",True
306277,502.091383,502.430612,so,word,kiksuya,so,"[(so, RB)]",True
334023,488.681859,488.951247,hadn't,word,lifereimagined,hadn't,"[(had, VBD), (n't, RB)]",True
615618,44.341723,44.760771,definitely,word,threemonths,definitely,"[(definitely, RB)]",True
523279,15.377551,15.726757,pretty,word,sweetaspie,pretty,"[(pretty, RB)]",True
388175,14.392612,14.658206,then,word,myfirstdaywiththeyankees,then,"[(then, RB)]",True


In [17]:
print(df_orthographic.shape)
df_orthographic['is_adverb'] = df_orthographic['pos_tagging'].apply(lambda x: 'RB' in str(x))
df_orthographic = df_orthographic.query("is_adverb == True").copy()
print(df_orthographic.shape)

(178831, 7)
(13149, 8)


In [18]:
df_orthographic['lemma'].value_counts()

lemma
so                1452
just               770
up                 700
there              675
not                515
                  ... 
busily               1
knowingly            1
highly               1
asymmetrically       1
satisfyingly         1
Name: count, Length: 392, dtype: int64

In [21]:
df_orthographic.sample(5)

,start,stop,name,tier,source,lemma,pos_tagging,is_adverb
90154,217.069161,217.448299,actually,word,bluehope,actually,"[(actually, RB)]",True
63684,289.097886,289.175283,when,word,becomingindian,when,"[(when, WRB)]",True
5848,102.280045,102.559410,wasn't,word,adollshouse,wasn't,"[(was, VBD), (n't, RB)]",True
23300,569.208844,569.388435,there,word,afatherscover,there,"[(there, RB)]",True
203128,646.932200,647.191610,enough,word,fromboyhoodtofatherhood,enough,"[(enough, RB)]",True
